<a href="https://www.kaggle.com/code/thaian36/notebook547394606b?scriptVersionId=349151516" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# DSC 2026 — Task 1 LegalIR | Kaggle T4 ×2 smoke

Notebook này chỉ chạy preflight và smoke tối đa 10 optimizer updates; không chạy full training hoặc publish Hugging Face. `public-official.json` không được đọc để train/eval/chọn checkpoint.

Trước khi chạy: attach **dataset LegalIR** và **một** source đã giải nén *hoặc* ZIP bundle. Nếu attach nhiều bản source, điền `SOURCE_ROOT` chính xác ở cell tiếp theo.

In [1]:
import os, json, hashlib, shutil, subprocess, sys, zipfile
from pathlib import Path

# Có thể điền trực tiếp các đường dẫn Kaggle; không điền URL GitHub vào trường Secret name.
# Để trống SOURCE_ROOT/SOURCE_ZIP thì cell tự tìm đúng *một* source có đủ signature.
SOURCE_ROOT = os.environ.get('LEGALIR_SOURCE_ROOT', '').strip()
SOURCE_ZIP = os.environ.get('LEGALIR_SOURCE_ZIP', '').strip()
DATA_ROOT = os.environ.get('LEGALIR_DATA_ROOT', '').strip()
OUTPUT_DIR = Path(os.environ.get('LEGALIR_OUTPUT_DIR', '/kaggle/working/legalir_kaggle_t4_smoke'))
INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')

SIGNATURE = ('scripts/run_preflight.py', 'scripts/run_smoke_test.py', 'src/config.py', 'configs/legalir_kaggle_t4_smoke.yaml')

def inspect_dir(path):
    path = Path(path)
    missing = [item for item in SIGNATURE if not (path / item).is_file()]
    return {'path': str(path), 'kind': 'directory', 'accepted': not missing, 'missing_signature': missing}

def inspect_zip(path):
    path = Path(path)
    try:
        with zipfile.ZipFile(path) as zf:
            names = set(zf.namelist())
    except Exception as exc:
        return {'path': str(path), 'kind': 'zip', 'accepted': False, 'reason': f'invalid zip: {exc}'}
    marker = 'scripts/run_preflight.py'
    prefixes = sorted({name[:-len(marker)] for name in names if name.endswith(marker)})
    accepted = [prefix for prefix in prefixes if all(prefix + item in names for item in SIGNATURE)]
    return {'path': str(path), 'kind': 'zip', 'accepted': len(accepted) == 1, 'accepted_prefixes': accepted, 'reason': None if len(accepted) == 1 else 'zip must contain exactly one LegalIR source root'}

def extract_zip_once(path):
    detail = inspect_zip(path)
    if not detail['accepted']:
        raise RuntimeError(json.dumps(detail, ensure_ascii=False, indent=2))
    digest = hashlib.sha256(Path(path).read_bytes()).hexdigest()
    destination = WORK_ROOT / f'legalir_source_{digest[:12]}'
    if destination.exists():
        checked = inspect_dir(destination)
        if checked['accepted']:
            return destination
        raise RuntimeError(f'Existing extraction is invalid: {checked}. Choose another output path; this cell never deletes it.')
    staging = destination.with_name(destination.name + '.staging')
    with zipfile.ZipFile(path) as zf:
        for member in zf.infolist():
            member_path = Path(member.filename)
            if member_path.is_absolute() or '..' in member_path.parts:
                raise RuntimeError(f'Unsafe ZIP member: {member.filename}')
        zf.extractall(staging)
    root = staging / detail['accepted_prefixes'][0]
    checked = inspect_dir(root)
    if not checked['accepted']:
        raise RuntimeError(f'Extraction did not produce valid source: {checked}')
    root.rename(destination)
    if staging.exists(): staging.rmdir()
    return destination

candidates = []
if SOURCE_ROOT:
    detail = inspect_dir(SOURCE_ROOT); candidates.append(detail)
    if not detail['accepted']: raise RuntimeError(f'LEGALIR_SOURCE_ROOT is invalid:\n{json.dumps(detail, ensure_ascii=False, indent=2)}')
    repo_dir = Path(SOURCE_ROOT)
elif SOURCE_ZIP:
    detail = inspect_zip(SOURCE_ZIP); candidates.append(detail)
    if not detail['accepted']: raise RuntimeError(f'LEGALIR_SOURCE_ZIP is invalid:\n{json.dumps(detail, ensure_ascii=False, indent=2)}')
    repo_dir = extract_zip_once(SOURCE_ZIP)
else:
    seen = set()
    for marker in INPUT_ROOT.rglob('run_preflight.py'):
        root = marker.parent.parent
        if root not in seen:
            seen.add(root); candidates.append(inspect_dir(root))
    for archive in INPUT_ROOT.rglob('*.zip'):
        candidates.append(inspect_zip(archive))
    accepted = [item for item in candidates if item.get('accepted')]
    print('Source discovery diagnostics:\n' + json.dumps(candidates, ensure_ascii=False, indent=2))
    if len(accepted) == 0:
        raise RuntimeError('Không tìm thấy source LegalIR đã xác minh. Attach source/ZIP, hoặc đặt LEGALIR_SOURCE_ROOT / LEGALIR_SOURCE_ZIP.')
    if len(accepted) > 1:
        raise RuntimeError('Tìm thấy nhiều source hợp lệ. Không tự chọn bản đầu tiên; hãy đặt LEGALIR_SOURCE_ROOT thành một đường dẫn trong danh sách diagnostics.')
    repo_dir = Path(accepted[0]['path']) if accepted[0]['kind'] == 'directory' else extract_zip_once(accepted[0]['path'])

assert inspect_dir(repo_dir)['accepted'], repo_dir
if str(repo_dir).startswith('/kaggle/input'):
    copied = WORK_ROOT / 'legalir_source_worktree'
    if copied.exists() and not inspect_dir(copied)['accepted']:
        raise RuntimeError(f'{copied} exists but is not valid source; choose another worktree path.')
    if not copied.exists(): shutil.copytree(repo_dir, copied)
    repo_dir = copied
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert not str(OUTPUT_DIR).startswith('/kaggle/input'), 'OUTPUT_DIR must be writable, not /kaggle/input'
os.chdir(repo_dir)
os.environ['PYTHONPATH'] = f'{repo_dir}{os.pathsep}' + os.environ.get('PYTHONPATH', '')
print('Source ready:', repo_dir)
print('Output directory:', OUTPUT_DIR)

Source discovery diagnostics:
[
  {
    "path": "/kaggle/input/datasets/thaian36/ggchampion/legalir_dsc2026_bundle_retrieval_smoke_hotfix2 (1)",
    "kind": "directory",
    "accepted": true,
    "missing_signature": []
  }
]
Source ready: /kaggle/working/legalir_source_worktree
Output directory: /kaggle/working/legalir_kaggle_t4_smoke


In [2]:
# Không cài torch / torchvision / torchaudio: Kaggle đã cung cấp bộ CUDA khớp nhau.
# Nếu pip yêu cầu restart kernel, restart rồi chạy lại từ cell Source/config.
completed = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--upgrade-strategy', 'only-if-needed', '-r', 'requirements-kaggle.txt'], check=False)
if completed.returncode: raise RuntimeError(f'pip failed with exit code {completed.returncode}')
check_code = '''import torch, transformers, peft, accelerate, pyarrow, bm25s, pyvi
print({'torch':torch.__version__, 'cuda':torch.version.cuda, 'transformers':transformers.__version__, 'peft':peft.__version__, 'accelerate':accelerate.__version__, 'pyarrow':pyarrow.__version__})'''
completed = subprocess.run([sys.executable, '-c', check_code], text=True, capture_output=True, check=False)
print(completed.stdout); print(completed.stderr)
if completed.returncode: raise RuntimeError('Dependency imports failed. Restart the Kaggle kernel once; if still failing, restore Kaggle torch runtime and do not pip-install torch.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 150.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 149.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.1/411.1 kB 315.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 96.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 347.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 138.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.5/47.5 kB 236.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 191.8 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


{'torch': '2.10.0+cu128', 'cuda': '12.8', 'transformers': '4.57.6', 'peft': '0.15.2', 'accelerate': '1.13.0', 'pyarrow': '21.0.0'}




In [3]:
def valid_data_root(path):
    path = Path(path)
    flat = all((path / name).is_file() for name in ('documents.parquet', 'chunks.parquet', 'queries_train.parquet', 'qrels_train.parquet'))
    nested = all((path / 'canonical' / name).is_file() for name in ('documents.parquet', 'chunks.parquet', 'queries_train.parquet', 'qrels_train.parquet'))
    return flat or nested

if DATA_ROOT:
    dataset_root = Path(DATA_ROOT)
    if not valid_data_root(dataset_root): raise RuntimeError(f'LEGALIR_DATA_ROOT is not a LegalIR dataset root: {dataset_root}')
else:
    data_candidates = []
    for document in INPUT_ROOT.rglob('documents.parquet'):
        candidate = document.parent.parent if document.parent.name == 'canonical' else document.parent
        if valid_data_root(candidate): data_candidates.append(candidate)
    data_candidates = sorted(set(data_candidates))
    print('Dataset candidates:', *map(str, data_candidates), sep='\n- ')
    if len(data_candidates) != 1:
        raise RuntimeError('Cần đúng một dataset LegalIR hoặc đặt LEGALIR_DATA_ROOT. Không nhầm source/ZIP/output với dataset.')
    dataset_root = data_candidates[0]
print('Dataset root:', dataset_root)
print('Dataset layout:', 'flat' if (dataset_root/'documents.parquet').is_file() else 'canonical/')
for item in sorted(dataset_root.iterdir()): print(item.name)

Dataset candidates:
- /kaggle/input/datasets/phucdangg/legalir-task1-clean-data
Dataset root: /kaggle/input/datasets/phucdangg/legalir-task1-clean-data
Dataset layout: flat
audit_report.json
chunks.parquet
documents.parquet
duplicate_groups.json
empty_context_ids.json
manifest.json
public-official.json
qrels_train.parquet
queries_train.parquet


In [4]:
# CPU preflight validates actual mounted schema and only reports BLOCKED for missing stage artifacts.
command = [sys.executable, 'scripts/run_preflight.py', '--config', 'configs/legalir_kaggle_t4_smoke.yaml', '--mode', 'smoke', '--set', f'data.dataset_root={dataset_root}', '--set', f'experiment.output_dir={OUTPUT_DIR}']
completed = subprocess.run(command, text=True, capture_output=True, check=False)
print('EXIT CODE:', completed.returncode)
print('--- STDOUT ---\n', completed.stdout, '\n--- STDERR ---\n', completed.stderr)
preflight_path = OUTPUT_DIR / 'preflight_report.json'
if not preflight_path.is_file(): raise RuntimeError('Preflight did not write report; inspect STDERR above before continuing.')
preflight = json.loads(preflight_path.read_text())
print(json.dumps({'overall': preflight['status'], 'checks': preflight['checks'], 'column_mappings': preflight['dataset_validation'].get('column_mappings')}, ensure_ascii=False, indent=2))

EXIT CODE: 2
--- STDOUT ---
 Preflight BLOCKED: /kaggle/working/legalir_kaggle_t4_smoke/preflight_report.json
 
--- STDERR ---
 
{
  "overall": "BLOCKED",
  "checks": [
    {
      "name": "dataset_validation",
      "status": "PASS"
    },
    {
      "name": "source_git_sha",
      "status": "BLOCKED",
      "reason": "source is not a Git checkout with a resolved commit; archive/source validation is possible but the workflow gate cannot pass"
    },
    {
      "name": "reranker_model_revision",
      "status": "BLOCKED",
      "reason": "an immutable Hugging Face commit SHA (40 hexadecimal characters) is required; do not use the moving default revision/main",
      "revision": null
    },
    {
      "name": "reranker_training_pairs",
      "status": "BLOCKED",
      "reason": "reranker_pairs.parquet is required by this training profile; receive the published training pairs rather than mining/relabeling in this notebook",
      "checked_paths": [
        "/kaggle/input/datasets/phuc

In [5]:
# Không dùng !nvidia-smi để tránh os.fork/JAX warning. Warning đó không phải CUDA error, nhưng subprocess sạch hơn.
completed = subprocess.run(['nvidia-smi'], text=True, capture_output=True, check=False)
print(completed.stdout); print(completed.stderr)
import torch
gpu_ok = torch.cuda.is_available() and torch.cuda.device_count() == 2
hardware = [(i, torch.cuda.get_device_name(i), torch.cuda.get_device_properties(i).total_memory) for i in range(torch.cuda.device_count())] if torch.cuda.is_available() else []
print({'target_two_gpu_available': gpu_ok, 'gpus': hardware})
if preflight['status'] != 'PASS':
    print('SMOKE SKIPPED: preflight is', preflight['status'], '. Resolve only the BLOCKED/FAIL checks shown above; kaggle_smoke_report.json is not expected yet.')
elif not gpu_ok:
    print('SMOKE NOT_RUN: this notebook needs exactly 2 visible GPUs for the target T4×2 diagnostic.')
else:
    command = [sys.executable, 'scripts/run_smoke_test.py', '--config', 'configs/legalir_kaggle_t4_smoke.yaml', '--dataset-root', str(dataset_root), '--num-processes', '2']
    completed = subprocess.run(command, text=True, capture_output=True, check=False)
    print('SMOKE EXIT CODE:', completed.returncode)
    print(completed.stdout); print(completed.stderr)
    if completed.returncode != 0: print('Smoke did not pass; inspect output above and the report if it exists. Do not package it as PASS.')

Sat Sep 12 01:24:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
smoke_path = OUTPUT_DIR / 'kaggle_smoke_report.json'
if not smoke_path.is_file():
    print('NOT_RUN: kaggle_smoke_report.json chưa tồn tại. Điều này bình thường khi preflight BLOCKED/FAIL hoặc smoke runner chưa hoàn thành.')
else:
    smoke = json.loads(smoke_path.read_text())
    print(json.dumps({'status': smoke['status'], 'checks': smoke['checks'], 'training': smoke.get('training'), 'vram': smoke.get('vram'), 'reload': smoke.get('reload')}, ensure_ascii=False, indent=2))
    if smoke['status'] == 'PASS':
        package = subprocess.run([sys.executable, 'scripts/build_bundle.py', '--run-dir', str(OUTPUT_DIR), '--dataset-validation', str(OUTPUT_DIR/'dataset_validation_report.json'), '--smoke-report', str(smoke_path)], text=True, capture_output=True, check=False)
        print(package.stdout); print(package.stderr)
        if package.returncode: raise RuntimeError(f'Packaging failed: exit={package.returncode}')
    else:
        print('Không package/publish: smoke report không PASS.')

NOT_RUN: kaggle_smoke_report.json chưa tồn tại. Điều này bình thường khi preflight BLOCKED/FAIL hoặc smoke runner chưa hoàn thành.
